In [1]:
%matplotlib inline
%load_ext autoreload

%autoreload 2

In [2]:
from pathlib import Path
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np
from scipy import ndimage
import scipy.signal as signal
import pickle
import math
import torch

from notebooks.imports import *
from src.utils import pmf_utils


### Config

In [3]:
from config import dir_config

raw_dir = Path(dir_config.data.raw)
processed_dir = Path(dir_config.data.processed)

#### Calculate and Save mean motion energy

In [ ]:
# def load_motion(filepath):
# 	motion_energy = torch.load(filepath)  # Shape: (channels, height, width)

# 	_, h, w = motion_energy.shape
# 	h_half = h // 2
# 	w_half = w // 2

# 	# Define center region indices (same size as quadrant, centered)
# 	h_start_c = (h - h_half) // 2
# 	h_end_c = h_start_c + h_half
# 	w_start_c = (w - w_half) // 2
# 	w_end_c = w_start_c + w_half

# 	# Compute motion energies using PyTorch mean
# 	me_global = motion_energy.mean(dim=(1, 2))
# 	me_center = motion_energy[:, h_start_c:h_end_c, w_start_c:w_end_c].mean(dim=(1, 2))

# 	me_first_quadrant = motion_energy[:, 0:h_half, 0:w_half].mean(dim=(1, 2))
# 	me_second_quadrant = motion_energy[:, 0:h_half, w_half:w].mean(dim=(1, 2))
# 	me_third_quadrant = motion_energy[:, h_half:h, 0:w_half].mean(dim=(1, 2))
# 	me_fourth_quadrant = motion_energy[:, h_half:h, w_half:w].mean(dim=(1, 2))

# 	return {"global": me_global, "center": me_center, "quadrants": {"Q1": me_first_quadrant, "Q2": me_second_quadrant, "Q3": me_third_quadrant, "Q4": me_fourth_quadrant}}


# import logging
# from concurrent.futures import ProcessPoolExecutor, as_completed
# from pathlib import Path
# import pickle
# from datetime import datetime

# # Set up logging
# logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
# logger = logging.getLogger()


# def process_motion_files(processed_dir, max_workers=16):
# 	motion_energy_dir = Path(processed_dir) / "motion_energy_arrays"
# 	motion_files = sorted(motion_energy_dir.glob("*.pt"))  # Optional: ensure consistent ordering

# 	motion_data_list = [None] * len(motion_files)
# 	motion_file_names = [fp.stem for fp in motion_files]

# 	with ProcessPoolExecutor(max_workers=max_workers) as executor:
# 		futures = {executor.submit(load_motion, fp): idx for idx, fp in enumerate(motion_files)}

# 		for i, future in enumerate(as_completed(futures), 1):
# 			idx = futures[future]
# 			try:
# 				motion_data_list[idx] = future.result()
# 			except Exception as e:
# 				logger.error(f"❌ Failed to load file {motion_files[idx]}: {e}")

# 			if i % 10 == 0 or i == len(motion_files):
# 				logger.info(f"📥 Loaded {i}/{len(motion_files)} files")

# 	motion_data_dict = {int(name.replace("motion_", "")): data for name, data in zip(motion_file_names, motion_data_list) if data is not None}

# 	# Save the motion data dictionary to a pickle file with a timestamp
# 	timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# 	pickle_filename = processed_dir / f"motion_data_dict_{timestamp}.pkl"

# 	with open(pickle_filename, "wb") as f:
# 		pickle.dump(motion_data_dict, f)

# 	logger.info(f"✅ Saved motion data to {pickle_filename}")


# # Call the function with your processed directory path
# process_motion_files(processed_dir)



In [ ]:
# import logging
# from concurrent.futures import ProcessPoolExecutor, as_completed
# from pathlib import Path
# import pickle
# from datetime import datetime

# # Set up logging
# logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
# logger = logging.getLogger()


# def process_motion_files(processed_dir, max_workers=16):
# 	motion_energy_dir = Path(processed_dir) / "motion_energy_arrays"
# 	motion_files = sorted(motion_energy_dir.glob("*.pt"))  # Optional: ensure consistent ordering

# 	motion_data_list = [None] * len(motion_files)
# 	motion_file_names = [fp.stem for fp in motion_files]

# 	with ProcessPoolExecutor(max_workers=max_workers) as executor:
# 		futures = {executor.submit(load_motion, fp): idx for idx, fp in enumerate(motion_files)}

# 		for i, future in enumerate(as_completed(futures), 1):
# 			idx = futures[future]
# 			try:
# 				motion_data_list[idx] = future.result()
# 			except Exception as e:
# 				logger.error(f"❌ Failed to load file {motion_files[idx]}: {e}")

# 			if i % 10 == 0 or i == len(motion_files):
# 				logger.info(f"📥 Loaded {i}/{len(motion_files)} files")

# 	motion_data_dict = {int(name.replace("motion_", "")): data for name, data in zip(motion_file_names, motion_data_list) if data is not None}

# 	# Save the motion data dictionary to a pickle file with a timestamp
# 	timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# 	pickle_filename = processed_dir / f"motion_data_dict_{timestamp}.pkl"

# 	with open(pickle_filename, "wb") as f:
# 		pickle.dump(motion_data_dict, f)

# 	logger.info(f"✅ Saved motion data to {pickle_filename}")


# # Call the function with your processed directory path
# process_motion_files(processed_dir)


---
---

In [4]:
import logging
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
import pickle
from datetime import datetime
import torch
import gc  # To clear unused memory
import os

# Set up logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger()


def load_motion(filepath):
	try:
		motion_energy = torch.load(filepath)  # Shape: (channels, height, width)

		_, h, w = motion_energy.shape
		h_half = h // 2
		w_half = w // 2

		# Define center region indices (same size as quadrant, centered)
		h_start_c = (h - h_half) // 2
		h_end_c = h_start_c + h_half
		w_start_c = (w - w_half) // 2
		w_end_c = w_start_c + w_half

		# Compute motion energies using PyTorch mean
		me_global = motion_energy.mean(dim=(1, 2))
		me_center = motion_energy[:, h_start_c:h_end_c, w_start_c:w_end_c].mean(dim=(1, 2))

		me_first_quadrant = motion_energy[:, 0:h_half, 0:w_half].mean(dim=(1, 2))
		me_second_quadrant = motion_energy[:, 0:h_half, w_half:w].mean(dim=(1, 2))
		me_third_quadrant = motion_energy[:, h_half:h, 0:w_half].mean(dim=(1, 2))
		me_fourth_quadrant = motion_energy[:, h_half:h, w_half:w].mean(dim=(1, 2))

		return {"global": me_global, "center": me_center, "quadrants": {"Q1": me_first_quadrant, "Q2": me_second_quadrant, "Q3": me_third_quadrant, "Q4": me_fourth_quadrant}}
	except Exception as e:
		logger.error(f"Failed to process {filepath}: {e}")
		return None


def process_batch(batch_files):
	motion_data_list = []
	for file in batch_files:
		data = load_motion(file)
		if data is not None:
			motion_data_list.append(data)
	return motion_data_list


In [ ]:



# def process_motion_files(processed_dir, max_workers=16, batch_size=10):
# 	motion_energy_dir = Path(processed_dir) / "motion_energy_arrays"
# 	motion_files = sorted(motion_energy_dir.glob("*.pt"))  # Optional: ensure consistent ordering

# 	# Split the files into batches to process
# 	file_batches = [motion_files[i : i + batch_size] for i in range(0, len(motion_files), batch_size)]

# 	motion_data_list = []
# 	motion_file_names = [fp.stem for fp in motion_files]

# 	# Set up a process pool for parallel batch processing
# 	with ProcessPoolExecutor(max_workers=max_workers) as executor:
# 		futures = {executor.submit(process_batch, batch): idx for idx, batch in enumerate(file_batches)}

# 		for i, future in enumerate(as_completed(futures), 1):
# 			idx = futures[future]
# 			try:
# 				result = future.result()
# 				motion_data_list.extend(result)  # Collect results from the batch
# 			except Exception as e:
# 				logger.error(f"❌ Failed to process batch {idx}: {e}")

# 			# Periodic progress logging
# 			if i % 5 == 0 or i == len(futures):
# 				logger.info(f"📥 Processed {i}/{len(futures)} batches")

# 			# Free up memory periodically
# 			gc.collect()

# 	# Filter out None results and map file names to data
# 	motion_data_dict = {int(name.replace("motion_", "")): data for name, data in zip(motion_file_names, motion_data_list) if data is not None}

# 	# Save the motion data dictionary to a pickle file with a timestamp
# 	timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# 	pickle_filename = Path(processed_dir) / f"motion_data_dict_{timestamp}.pkl"

# 	with open(pickle_filename, "wb") as f:
# 		pickle.dump(motion_data_dict, f)

# 	logger.info(f"✅ Saved motion data to {pickle_filename}")


# process_motion_files(processed_dir)


In [6]:
import torch
import gc
import pickle
from pathlib import Path
from datetime import datetime


def process_motion_files(processed_dir, save_every=100):
	motion_energy_dir = Path(processed_dir) / "motion_energy_arrays"
	motion_files = sorted(motion_energy_dir.glob("*.pt"))

	motion_data_dict = {}
	files_to_delete = []  # Keep track of files to delete after saving
	processed_count = 0

	timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
	base_filename = processed_dir / f"motion_data_dict_{timestamp}"

	for idx, motion_file in enumerate(motion_files, 1):
		try:
			# Extract the number from the file name and use it as the key
			file_stem = motion_file.stem
			# Attempt to extract the numeric part from the file name (after 'me_array_')
			file_idx = int(file_stem.replace("me_array_", ""))

			# Load the motion data
			data = load_motion(motion_file)
			motion_data_dict[file_idx] = data
			files_to_delete.append(motion_file)  # Mark for deletion after save
		except ValueError as e:
			print(f"❌ Failed to extract file index from {motion_file.stem}: {e}")
			continue
		except Exception as e:
			print(f"❌ Failed to load file {motion_file}: {e}")
			continue

		# Free memory manually
		del data
		torch.cuda.empty_cache()
		gc.collect()

		if idx % 100 == 0 or idx == len(motion_files):
			print(f"📥 Processed {idx}/{len(motion_files)} files")

		# Save every `save_every` files or at end
		if idx % save_every == 0 or idx == len(motion_files):
			checkpoint_filename = f"{base_filename}_part_{(idx // save_every)}.pkl"
			with open(checkpoint_filename, "wb") as f:
				pickle.dump(motion_data_dict, f)
			print(f"✅ Saved checkpoint: {checkpoint_filename}")

			# Now safely delete files that were saved
			for file in files_to_delete:
				try:
					file.unlink()
					print(f"🗑️ Deleted file: {file}")
				except Exception as e_del:
					print(f"❌ Failed to delete file {file}: {e_del}")

			# Clear memory
			motion_data_dict.clear()
			files_to_delete.clear()

		processed_count += 1


process_motion_files(processed_dir, save_every=100)


📥 Processed 100/113874 files
✅ Saved checkpoint: /mnt/data/processed/motion_data_dict_20250426_184227_part_1.pkl
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_00000.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_00001.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_00002.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_00003.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_00004.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_00005.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_00006.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_00007.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_00008.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_00009.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_00010.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_

2025-04-27 02:16:18,870 - ERROR - Failed to process /mnt/data/processed/motion_energy_arrays/me_array_112841.pt: PytorchStreamReader failed reading zip archive: failed finding central directory


📥 Processed 24200/113874 files
✅ Saved checkpoint: /mnt/data/processed/motion_data_dict_20250426_184227_part_242.pkl
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_112818.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_112819.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_11282.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_112820.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_112821.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_112822.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_112823.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_112824.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_112825.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_112826.pt
🗑️ Deleted file: /mnt/data/processed/motion_energy_arrays/me_array_112827.pt
🗑️ Deleted file: /mnt/data/processed/

In [4]:
import pickle
from pathlib import Path


def compile_motion_data(processed_dir, compiled_filename="motion_data_compiled.pkl"):
	processed_dir = Path(processed_dir)
	part_files = sorted(processed_dir.glob("motion_data_dict_*_part_*.pkl"))

	all_motion_data = {}

	for part_file in part_files:
		print(f"📦 Loading {part_file.name}...")
		with open(part_file, "rb") as f:
			part_data = pickle.load(f)
			all_motion_data.update(part_data)  # Merge into one dict

	# Save the compiled dictionary
	compiled_path = processed_dir / compiled_filename
	with open(compiled_path, "wb") as f:
		pickle.dump(all_motion_data, f)

	print(f"✅ Compiled {len(part_files)} parts into {compiled_path} ({len(all_motion_data)} entries total)")

	return compiled_path

compile_motion_data(processed_dir)


📦 Loading motion_data_dict_20250426_184227_part_1.pkl...
📦 Loading motion_data_dict_20250426_184227_part_10.pkl...
📦 Loading motion_data_dict_20250426_184227_part_100.pkl...
📦 Loading motion_data_dict_20250426_184227_part_1000.pkl...
📦 Loading motion_data_dict_20250426_184227_part_1001.pkl...
📦 Loading motion_data_dict_20250426_184227_part_1002.pkl...
📦 Loading motion_data_dict_20250426_184227_part_1003.pkl...
📦 Loading motion_data_dict_20250426_184227_part_1004.pkl...
📦 Loading motion_data_dict_20250426_184227_part_1005.pkl...
📦 Loading motion_data_dict_20250426_184227_part_1006.pkl...
📦 Loading motion_data_dict_20250426_184227_part_1007.pkl...
📦 Loading motion_data_dict_20250426_184227_part_1008.pkl...
📦 Loading motion_data_dict_20250426_184227_part_1009.pkl...
📦 Loading motion_data_dict_20250426_184227_part_101.pkl...
📦 Loading motion_data_dict_20250426_184227_part_1010.pkl...
📦 Loading motion_data_dict_20250426_184227_part_1011.pkl...
📦 Loading motion_data_dict_20250426_184227_part

PosixPath('/mnt/data/processed/motion_data_compiled.pkl')